In [ ]:
import json
import sys

from pathlib import Path

current = Path().resolve().parent.parent.parent
source_code = current / "src"

sys.path.append(str(source_code))

from job_tracker.infrastructure import PathResolver
from job_tracker.orchestration import DistributionChart

In [ ]:
path_resolver = PathResolver(base_path=current)
distribution_chart = DistributionChart()

data_path = path_resolver.finalized_directory()

In [ ]:
json_data = []

for file in data_path.rglob("*.json"):
    with open(file, "r", encoding="utf-8") as f:
        json_data.append(json.load(f))

all_length = len(json_data)

print(all_length)

In [ ]:
all_keys = json_data[1].keys()
all_keys

In [ ]:
all_data = dict()

for key in all_keys:
    context_doc = dict()
    for context in json_data[1][key].keys():
        context_doc[context] = ""
    all_data[key] = context_doc

In [ ]:
job_keys = {}
prev_keys = None

for i, item in enumerate(json_data):
    current_keys = list(item["application"].keys())

    if current_keys != prev_keys:
        job_keys[i] = current_keys
        prev_keys = current_keys

job_keys

In [ ]:
json_data[18]["application"]["timeline"]

In [ ]:
json_data[18]['application']

In [ ]:
applications = []

for idx, data in enumerate(json_data):
    app = data.get("application", {})

    if idx != 18 and idx < 28:
        status = app.get("status")
        status = "unknown" if status in ("unknown", "open") else status

        result = {
            "current_status": status,
            "timeline": [{
                "status": status,
                "date": app.get("applied_at")
            }]
        }
    else:
        result = app

    applications.append(result)

# Current Status

In [ ]:
aplication_current_status = [
    x['current_status'] 
    for x in applications
]
aplication_current_status

In [ ]:
from typing import Any, Dict, List, Optional, Sequence
from collections import Counter
from pydantic import BaseModel, Field

class LevelDataSchema(BaseModel):
    values: List[str] = Field(
        default_factory=list,
        description="List of category labels or level names."
    )
    
    sizes: List[int] = Field(
        default_factory=list,
        description="List of counts corresponding to each value."
    )

    parents: list[str] = Field(
        default_factory=list,
        description="List of category parents for value."
    )


class HirarchyRuleSchema(BaseModel):
    base_level: List[str] = Field(default_factory=lambda: [""])
    level_one: List[str] = Field(default_factory=lambda: ["GET"])
    level_two: List[str] = Field(default_factory=lambda: ["unknown", "applied"])
    applied: List[str] = Field(default_factory=lambda: ["interview", "offered", "rejected", "dropped"])
    unknown: List[str] = Field(default_factory=list)


def filter_based_on_rule(data, rule):
    result = [
        x
        for x in data
        if x in rule
    ]

    return result


def hirarchy_process(labels, sizes, settings):
    sub_result = ["level_one", "level_two", "level_three"]

    result=dict()

    base = {
        "values": list(),
        "sizes": list(),
        "parents":list(),
    }

    for sub in sub_result:
        result[sub] = base

    result[sub_result[0]]["values"] = settings.level_one
    result[sub_result[0]]["parents"] = settings.base_level * len(result[sub_result[0]]["values"])

    result[sub_result[1]]["values"] = filter_based_on_rule(labels, settings.level_two)
    result[sub_result[1]]["parents"] = 

    result[sub_result[2]]["values"] = filter_based_on_rule(labels, settings.level_one)
    result[sub_result[2]]["parents"] = settings.base_level * len(result[sub_result[0]]["values"])

    return result


def hirarchy_rule(
    base_level: Optional[List[str]] = None,
    level_one: Optional[List[str]] = None,
    level_two: Optional[List[str]] = None,
    level_three: Optional[List[str]] = None,
) -> Dict[str, List[str]]:
    
    rule = HirarchyRuleSchema()

    result= {
        "base_level": base_level,
        "level_one": level_one,
        "level_two": level_two,
        "applied": filter_based_on_rule(level_three, rule.applied),
        "unknown": filter_based_on_rule(level_three, rule.unknown),
    }

    return result


def hirarchy_gen(
    data:Sequence[str],
    base_level: List[str]= None,
    level_one: List[str] = None,
    level_two: List[str] = None,
    level_three: List[str] = None,
)-> Dict[str, LevelDataSchema]:
    counter_result = Counter(data)

    labels, sizes = counter_result.keys(), counter_result.values()

    settings = hirarchy_rule(
                    base_level=base_level,
                    level_one=level_one,
                    level_two=level_two,
                    level_three=level_three,
                )
    
    hirarchy_process(labels, sizes, settings)
    
    result[sub_result[0]] = {
        "values": filter_based_on_rule(labels, rule),
        "sizes": [

        ],
        "parents":[""]*len(main_parents),
    }

    result[sub_result[1]] = {
        "values": ,
        "sizes": ,
        "parents":,
    }

    result[sub_result[3]] = {
        "values": ,
        "sizes": ,
        "parents":,
    }
    
    

    get_total = sum(sizes)

    get_applied = sizes[0] + sizes[2] + sizes[3]

    
    result = {
        x:{
            "values": list(),
            "sizes": list(),
        }
        for x in sub_result
    }

    

    return result


def hirachy_viz_gen(
    level_one:LevelDataSchema,
    level_two:LevelDataSchema,
    level_three:LevelDataSchema,
)-> Dict[str, Sequence[Any]]:
    sub_result = ["labels", "parents", "sizes"]
    result = {x:list() for x in sub_result}

    result[sub_result[0]] = level_one.values + level_two.values + level_three.values

    result[sub_result[1]] = level_one.parents + level_two.parents + level_three.parents

    result[sub_result[3]] = level_one.sizes + level_two.parents + level_three.prents

    return result


def orchestration_shirachy(data: Sequence[Any]):
    hirarchy = hirarchy_gen(
                    data = data,
                    main_parents = ["GET"]
                )
    
    result = hirachy_viz_gen(
                level_one=hirarchy.level_one,
                level_two=hirarchy.level_two,
                level_three=hirarchy.level_three,
            )
    
    return result

In [ ]:
app_viz_status = orchestration_shirachy(data = aplication_current_status)

In [ ]:
distribution_chart.distribution_chart(
    names=app_viz_status.labels,
    parents=app_viz_status.parents,
    sizes=app_viz_status.sizes,
    chart_type="sunburst",
    title_text="Application Current Status",
)

# Timeline

In [ ]:
aplication_timeline = [
    x['timeline'] 
    for x in applications
]

aplication_timeline